In [ ]:
# CHECKPOINT 6A — Independent 13-window explanation stability consolidation
# Uses existing 5A–5H evidence only; does NOT run explainers, fit models,
# change thresholds, or modify historical/experiment batch files.

from pathlib import Path
from itertools import combinations
import hashlib
import importlib.metadata as im
import json
import numpy as np
import pandas as pd

# 0. Read-only prerequisites and frozen protocol.
assert im.version('scikit-learn') == '1.6.1'
assert im.version('shap') == '0.54.1'
assert im.version('lime') == '0.2.0.1'
assert im.version('numpy') == '2.4.6'
assert im.version('pandas') == '2.2.3'
assert 'ROOT' in globals(), 'STOP: ROOT from verified Colab session missing'

ROOT = Path(ROOT)
BASE = ROOT / 'SUPERVISOR_FINAL_EXPERIMENTS_2026'
EVIDENCE = BASE / '06_explanation_evaluation'
FROZEN_HASH = '849e9d7ef15cc3b135ac74854f491ae3c2113f39c3ef527ca789cde8e829334c'
SEEDS = [42, 123, 456, 789, 2026]
EXPECTED_WINDOWS = (
    [f'UKDALE_{i:02d}' for i in range(1, 6)] +
    [f'AMPDS2_{i:02d}' for i in range(1, 9)]
)
FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff', 'median',
    'min', 'n_points', 'peak_count', 'q25', 'q75', 'range',
    'rms', 'skew', 'slope', 'std', 'std_diff', 'sum', 'var'
]
COLUMNS = [f'{name}_z' for name in FEATURES]

protocol_file = BASE / '01_protocol' / 'CHECKPOINT4B_STABILITY_13WINDOW_849e9d7e.json'
assert protocol_file.is_file(), 'STOP: preserved protocol missing'
protocol_bytes = protocol_file.read_bytes()
assert hashlib.sha256(protocol_bytes).hexdigest() == FROZEN_HASH, 'STOP: protocol hash differs'
p = json.loads(protocol_bytes)
assert p['seeds'] == SEEDS
assert p['feature_order'] == FEATURES
assert p['target'] == '-IsolationForest.decision_function(X)'
assert p['kernel_shap']['nsamples'] == 300
assert p['kernel_shap']['coalition_rng_seed'] == 42
assert p['kernel_shap']['l1_reg'] == 'num_features(10)'
assert p['lime']['mode'] == 'regression'
assert p['lime']['num_samples'] == 5000
assert p['lime']['reference_size'] == 8192
assert p['evaluation']['top_k'] == 5

D5_PATH = (Path(D5) if 'D5' in globals() else
           EVIDENCE / 'D5_C3_12_C4_12_LOCAL_XAI_EVIDENCE_20261106T183849_627223Z')
selected = pd.read_csv(D5_PATH / 'selected_scaled_inputs_13_windows.csv', float_precision='round_trip')
assert len(selected) == 13 and selected['window_id'].is_unique
assert set(selected['window_id']) == set(EXPECTED_WINDOWS)
assert selected['dataset'].value_counts().to_dict() == {'UK-DALE': 5, 'AMPds2': 8}
selected = selected.set_index('window_id')

# 1. Frozen, explicit source inventory: 8 previously completed checkpoints.
# Each batch contains runs, per-seed 23-feature mappings, and pairwise Jaccard files.
BATCHES = [
    ('5A', 'SHAP', ['UKDALE_01'],
     'CHECKPOINT5A_UKDALE01_SHAP_BACKGROUND_5SEEDS_849e9d7e', 'ukdale01_shap'),
    ('5B', 'LIME', ['UKDALE_01'],
     'CHECKPOINT5B_UKDALE01_LIME_PERTURBATION_5SEEDS_849e9d7e', 'ukdale01_lime'),
    ('5C', 'SHAP', ['AMPDS2_01'],
     'CHECKPOINT5C_AMPDS201_SHAP_BACKGROUND_5SEEDS_849e9d7e', 'ampds201_shap'),
    ('5D', 'LIME', ['AMPDS2_01'],
     'CHECKPOINT5D_AMPDS201_LIME_PERTURBATION_5SEEDS_849e9d7e', 'ampds201_lime'),
    ('5E', 'SHAP', [f'UKDALE_{i:02d}' for i in range(2, 6)],
     'CHECKPOINT5E_UKDALE_REMAINING4_SHAP_5SEEDS_849e9d7e', 'ukdale_remaining4_shap'),
    ('5F', 'LIME', [f'UKDALE_{i:02d}' for i in range(2, 6)],
     'CHECKPOINT5F_UKDALE_REMAINING4_LIME_PERTURBATION_5SEEDS_849e9d7e', 'ukdale_remaining4_lime'),
    ('5G', 'SHAP', [f'AMPDS2_{i:02d}' for i in range(2, 9)],
     'CHECKPOINT5G_AMPDS2_REMAINING7_SHAP_5SEEDS_849e9d7e', 'ampds2_remaining7_shap'),
    ('5H', 'LIME', [f'AMPDS2_{i:02d}' for i in range(2, 9)],
     'CHECKPOINT5H_AMPDS2_REMAINING7_LIME_PERTURBATION_5SEEDS_849e9d7e', 'ampds2_remaining7_lime'),
]

def file_hash(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''):
            h.update(b)
    return h.hexdigest()

all_runs, all_features, all_pairs = [], [], []
source_files = {}
for stage, method, windows, directory, prefix in BATCHES:
    folder = EVIDENCE / directory
    assert folder.is_dir(), f'STOP: missing batch {stage} at {folder}'
    spec = {
        'runs': folder / f'{prefix}_runs.csv',
        'pairs': folder / f'{prefix}_pairwise_jaccard.csv',
        'features': folder / (
            f'{prefix}_23_features_by_seed.csv' if method == 'SHAP'
            else f'{prefix}_23_feature_mapping.csv'
        )
    }
    # Some single-window and UK-DALE batch mapping names differ from generic naming.
    if stage == '5B':
        spec['features'] = folder / 'ukdale01_lime_23_feature_mapping.csv'
    if stage == '5D':
        spec['features'] = folder / 'ampds201_lime_23_feature_mapping.csv'
    for kind, path in spec.items():
        assert path.is_file(), f'STOP: missing {kind} file at {path}'
        source_files[str(path.relative_to(BASE))] = file_hash(path)
    frames = {name: pd.read_csv(path, float_precision='round_trip')
              for name, path in spec.items()}
    # Canonicalize the IF anomaly-score field across source batches.
    # SHAP 5A/5C/5G used 'score'; SHAP 5E and all LIME used 'anomaly_score'.
    # This changes ONLY the in-memory consolidation representation.
    if 'score' in frames['runs'].columns:
        assert 'anomaly_score' not in frames['runs'].columns, (stage, 'ambiguous score columns')
        frames['runs'] = frames['runs'].rename(columns={'score': 'anomaly_score'})
    assert 'anomaly_score' in frames['runs'].columns, (stage, 'missing anomaly score')
    assert pd.to_numeric(frames['runs']['anomaly_score'], errors='coerce').notna().all(), (stage, 'invalid anomaly score')
    nr = 5 * len(windows)
    assert len(frames['runs']) == nr
    assert len(frames['features']) == nr * 23
    assert len(frames['pairs']) == 10 * len(windows)
    for kind, df in frames.items():
        assert set(df.window_id.astype(str)) == set(windows), f'STOP: {stage}/{kind} windows'
        assert df.window_id.notna().all()
        df.insert(0, 'source_checkpoint', stage)
        df.insert(1, 'method', method)
        df.insert(2, 'dataset', 'UK-DALE' if windows[0].startswith('UKDALE') else 'AMPds2') if 'dataset' not in df.columns else None
    all_runs.append(frames['runs'])
    all_features.append(frames['features'])
    all_pairs.append(frames['pairs'])
    print(f'PASS {stage}: {nr} {method} runs, {nr*23} feature records, {len(windows)*10} pairs')

runs = pd.concat(all_runs, ignore_index=True, sort=False)
features = pd.concat(all_features, ignore_index=True, sort=False)
pairs = pd.concat(all_pairs, ignore_index=True, sort=False)
assert (len(runs), len(features), len(pairs)) == (130, 2990, 260)
assert not runs.duplicated(['method', 'window_id', 'seed']).any()
assert not features.duplicated(['method', 'window_id', 'seed', 'feature_index']).any()
assert not pairs.duplicated(['method', 'window_id', 'seed_a', 'seed_b']).any()

# 2. Recalculate every top-five list and every pairwise Jaccard
# from original-feature mapping records. Do not trust saved summaries.
window_rows = []
for method in ['SHAP', 'LIME']:
    for win in EXPECTED_WINDOWS:
        rr = runs.loc[(runs.method == method) & (runs.window_id == win)]
        ff = features.loc[(features.method == method) & (features.window_id == win)]
        pp = pairs.loc[(pairs.method == method) & (pairs.window_id == win)]
        assert len(rr) == 5 and len(ff) == 115 and len(pp) == 10
        assert set(rr.seed.astype(int)) == set(SEEDS)
        assert set(ff.seed.astype(int)) == set(SEEDS)
        ds = 'UK-DALE' if win.startswith('UKDALE') else 'AMPds2'
        assert rr.dataset.eq(ds).all()
        top_sets = {}
        additivity = []
        for seed in SEEDS:
            run = rr.loc[rr.seed.eq(seed)]
            f = ff.loc[ff.seed.eq(seed)].sort_values('feature_index')
            assert len(run) == 1 and len(f) == 23
            run = run.iloc[0]
            assert list(f.feature_index.astype(int)) == list(range(23))
            assert list(f.feature) == FEATURES
            assert f.top5.sum() == 5
            values = f.scaled_feature_value.to_numpy(dtype=np.float64)
            expected_values = selected.loc[win, COLUMNS].to_numpy(dtype=np.float32).astype(np.float64)
            assert np.max(np.abs(values - expected_values)) <= 1e-7
            archived_score = float(selected.loc[win, 'anomaly_score'])
            target = float(run['anomaly_score'])  # canonicalized before concatenation
            assert abs(target - archived_score) <= 1e-9
            if method == 'SHAP':
                contribution = f.shap_value.to_numpy(dtype=float)
                assert np.isfinite(contribution).all()
                assert np.allclose(f.abs_shap_value.to_numpy(dtype=float), np.abs(contribution), atol=1e-12)
                assert int(np.count_nonzero(contribution)) == 10
                add = abs(float(run.expected_value) + float(contribution.sum()) - target)
                assert add <= 1e-6
                assert abs(add - float(run.additivity_residual)) <= 1e-9
                additivity.append(add)
                sort_idx = np.lexsort((np.arange(23), -np.abs(contribution)))[:5]
            else:
                assert f.selected_in_surrogate.sum() == 10
                weights = f.lime_weight.to_numpy(dtype=float)
                assert np.isfinite(weights[f.selected_in_surrogate.to_numpy(dtype=bool)]).all()
                assert np.isnan(weights[~f.selected_in_surrogate.to_numpy(dtype=bool)]).all()
                assert int(run.selected_feature_count) == 10
                assert np.isfinite(float(run.local_r2)) and float(run.local_r2) <= 1 + 1e-8
                assert abs(abs(float(run.surrogate_prediction) - target) - float(run.absolute_local_prediction_error)) <= 1e-9
                eligible = np.flatnonzero(f.selected_in_surrogate.to_numpy(dtype=bool))
                sort_idx = sorted(eligible, key=lambda idx: (-abs(weights[idx]), int(idx)))[:5]
            top_set = set(map(int, sort_idx))
            saved_set = set(f.loc[f.top5, 'feature_index'].astype(int))
            assert saved_set == top_set, f'STOP: top-five mapping mismatch: {method}/{win}/{seed}'
            assert set(str(run.top5_features).split('|')) == {FEATURES[i] for i in top_set}
            top_sets[seed] = top_set
        j_values = []
        expected_pairs = set(combinations(SEEDS, 2))
        seen_pairs = set()
        for _, row in pp.iterrows():
            a, b = int(row.seed_a), int(row.seed_b)
            assert (a, b) in expected_pairs and (a, b) not in seen_pairs
            seen_pairs.add((a, b))
            intersection = len(top_sets[a] & top_sets[b])
            union = len(top_sets[a] | top_sets[b])
            j = intersection / union
            assert int(row.intersection_count) == intersection
            assert int(row.union_count) == union
            assert abs(j - float(row.top5_jaccard)) <= 1e-12
            j_values.append(j)
        assert seen_pairs == expected_pairs
        result = {
            'dataset': ds,
            'method': method,
            'window_id': win,
            'n_seeds': 5,
            'n_pairs': 10,
            'mean_top5_jaccard': float(np.mean(j_values)),
            'min_top5_jaccard': float(np.min(j_values)),
            'max_top5_jaccard': float(np.max(j_values)),
            'max_additivity_residual': float(max(additivity)) if method == 'SHAP' else np.nan,
            'mean_lime_local_r2': float(rr.local_r2.mean()) if method == 'LIME' else np.nan,
            'seed_sd_lime_local_r2': float(rr.local_r2.std(ddof=1)) if method == 'LIME' else np.nan,
            'mean_lime_absolute_local_error': float(rr.absolute_local_prediction_error.mean()) if method == 'LIME' else np.nan,
        }
        window_rows.append(result)
window_df = pd.DataFrame(window_rows)
assert len(window_df) == 26

# 3. Summaries: the independent sampling units are windows, not seed-pairs.
summary_rows = []
for method in ['SHAP', 'LIME']:
    for ds in ['UK-DALE', 'AMPds2', 'Overall']:
        w = window_df.loc[window_df.method.eq(method)]
        if ds != 'Overall':
            w = w.loc[w.dataset.eq(ds)]
        n = len(w)
        assert n == {'UK-DALE':5, 'AMPds2':8, 'Overall':13}[ds]
        summary_rows.append({
            'method': method,
            'dataset': ds,
            'n_windows': n,
            'n_runs': n*5,
            'n_seed_pairs': n*10,
            'mean_top5_jaccard': float(w.mean_top5_jaccard.mean()),
            'sample_sd_top5_jaccard_across_windows': float(w.mean_top5_jaccard.std(ddof=1)),
            'mean_lime_r2_across_window_means': (float(w.mean_lime_local_r2.mean()) if method == 'LIME' else np.nan),
            'sample_sd_lime_r2_across_window_means': (float(w.mean_lime_local_r2.std(ddof=1)) if method == 'LIME' else np.nan),
            'mean_lime_abs_local_error_across_window_means': (float(w.mean_lime_absolute_local_error.mean()) if method == 'LIME' else np.nan)
        })
summary_df = pd.DataFrame(summary_rows)
assert len(summary_df) == 6

# Predefined consistency anchors calculated from existing saved evidence,
# NOT used for parameter selection or threshold fitting.
anchors = {
    ('SHAP','UK-DALE'): 1.0,
    ('SHAP','AMPds2'): 0.9666666666666667,
    ('SHAP','Overall'): 0.9794871794871796,
    ('LIME','UK-DALE'): 0.7923809523809524,
    ('LIME','AMPds2'): 0.85,
    ('LIME','Overall'): 0.827838827838828,
}
for _, row in summary_df.iterrows():
    assert abs(row.mean_top5_jaccard - anchors[(row.method,row.dataset)]) <= 1e-12

# 4. Only now preserve a distinct consolidated evidence package.
output = EVIDENCE / 'CHECKPOINT6A_13WINDOW_STABILITY_CONSOLIDATED_849e9d7e'

def csv_bytes(df):
    return df.to_csv(index=False, float_format='%.17g').encode('utf-8')

report_md = '# Residential XAI top-five stability — controlled reconstruction\n\n'
report_md += ('Protocol SHA-256: `' + FROZEN_HASH + '`\n\n'
              'Thirteen purposively selected residential test windows (5 UK-DALE; 8 AMPds2), five seeds per method. '
              'Kernel SHAP changes only the training-background sampling seed (coalition RNG held at 42); '
              'LIME changes only the perturbation seed (8192-row reference held fixed). '
              'Top-five Jaccard is computed for 10 seed pairs per window and averaged within each window. '
              'Reported sample SD is across windows, not across overlapping seed-pair comparisons.\n\n')
report_md += '| Dataset | Method | Windows | Mean Jaccard | Window-level sample SD |\n|---|---|---:|---:|---:|\n'
for _, r in summary_df.iterrows():
    report_md += (f'| {r.dataset} | {r.method} | {int(r.n_windows)} | '
                  f'{r.mean_top5_jaccard:.4f} | {r.sample_sd_top5_jaccard_across_windows:.4f} |\n')
report_md += ('\n**Limitations.** The selected windows are not a random residential population sample. '
              'The historical Kernel SHAP full-vector attribution identity was not reproduced, although its '
              'background expectation, archived additivity, and frozen Isolation Forest scores were verified. '
              'These are controlled reconstructions, not an exact historical SHAP rerun. '
              'Jaccard measures feature-set consistency, not correctness, anomaly truth, or causal validity. '
              'LIME local R-squared is local surrogate fidelity, not held-out predictive performance. '
              'No statistical significance test is implied. No model was retrained.\n')
manifest = {
    'study': '13_window_five_seed_SHAP_LIME_stability',
    'protocol_sha256': FROZEN_HASH,
    'historical_Kernel_SHAP_full_vector_identity': 'NOT_ESTABLISHED',
    'input_source_batches': [b[0] for b in BATCHES],
    'counts': {'windows':13, 'methods':2, 'runs':len(runs), 'feature_records':len(features), 'pair_records':len(pairs)},
    'source_sha256': source_files,
    'unit_of_summary': 'window',
    'jaccard_metric': 'mean_of_10_pairwise_top5_set_jaccards_per_window',
    'sd_rule': 'sample_standard_deviation_ddof1_across_window_means',
    'no_model_retraining': True,
    'historical_files_modified': False,
}
payloads = {
    'all_130_explanation_runs.csv': csv_bytes(runs),
    'all_2990_original_feature_records.csv': csv_bytes(features),
    'all_260_pairwise_jaccard.csv': csv_bytes(pairs),
    'window_level_26_method_records.csv': csv_bytes(window_df),
    'dataset_and_overall_summaries.csv': csv_bytes(summary_df),
    'evidence_manifest.json': (json.dumps(manifest, indent=2, sort_keys=True) + '\n').encode('utf-8'),
    'thesis_results_readme.md': report_md.encode('utf-8'),
}
for name, blob in payloads.items():
    dest = output / name
    if dest.exists():
        assert dest.read_bytes() == blob, f'STOP: Existing consolidation differs: {dest}'
output.mkdir(exist_ok=True)
for name, blob in payloads.items():
    dest = output / name
    if not dest.exists():
        with dest.open('xb') as stream:
            stream.write(blob)
    assert dest.read_bytes() == blob

print('\n=== CHECKPOINT 6A — FINAL RECONCILIATION ===')
print(summary_df.to_string(index=False, float_format=lambda v: f'{v:.6f}'))
print('\nPASS: 130 original explanation run records')
print('PASS: 2990 original-feature records; all top-five sets rederived')
print('PASS: 260 seed-pair Jaccards independently recomputed')
print('PASS: 26 window-by-method summaries, 6 grouped summaries')
print('PASS: Original protocol SHA-256 unchanged')
print('PASS: New consolidated evidence only; historical files unchanged')
print('Evidence directory:', output)
print('PASS: CHECKPOINT6A_ALL_13_WINDOWS_CONSOLIDATED')
